In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import Row

data_employee = [
    Row(employee_id=1, name='Alice', city='New York'),
    Row(employee_id=2, name='Bob', city='Los_Angeles'),
    Row(employee_id=3, name='Charlie', city='Chicago'),
    Row(employee_id=4, name='David', city='Houston'),
    Row(employee_id=5, name='Eva', city='Phoenix')
]
data_employee=spark.createDataFrame(data_employee)

data_employee.createOrReplaceTempView("data_employee")

display(data_employee)


employee_id,name,city
1,Alice,New York
2,Bob,Los_Angeles
3,Charlie,Chicago
4,David,Houston
5,Eva,Phoenix


In [0]:
%sql
create or replace table my_files.my_schema.emm_target_table
as
select *,current_timestamp() as insert_timestamp,
current_timestamp() as update_timestamp
from data_employee

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from my_files.my_schema.emm_target_table

employee_id,name,city,insert_timestamp,update_timestamp
1,Alice,New York,2026-03-07T12:10:49.292Z,2026-03-07T12:10:49.292Z
2,Bob,Los_Angeles,2026-03-07T12:10:49.292Z,2026-03-07T12:10:49.292Z
3,Charlie,Chicago,2026-03-07T12:10:49.292Z,2026-03-07T12:10:49.292Z
4,David,Houston,2026-03-07T12:10:49.292Z,2026-03-07T12:10:49.292Z
5,Eva,Phoenix,2026-03-07T12:10:49.292Z,2026-03-07T12:10:49.292Z


In [0]:
from pyspark.sql import SparkSession, Row



# Sample Employee info
data_employee = [
    Row(employee_id=1, name='Alice', city='Phoenix'),
    Row(employee_id=2, name='Bob', city='Houston'),
    Row(employee_id=6, name='Alfred', city='Phoenix'),
    Row(employee_id=7, name='Kate', city='Los Angeles')
]

# Create DataFrame from the list of Rows
df_employee = spark.createDataFrame(data_employee)

# Create a Temporary View to use Spark SQL queries
df_employee.createOrReplaceTempView('employee')


display(df_employee)

employee_id,name,city
1,Alice,Phoenix
2,Bob,Houston
6,Alfred,Phoenix
7,Kate,Los Angeles


In [0]:
%sql
MERGE INTO my_files.my_schema.emm_target_table target using employee source on target.employee_id=source.employee_id

WHEN MATCHED THEN 
UPDATE SET 
target.name=source.name,
target.city=source.city,
target.update_timestamp=current_timestamp()
WHEN NOT MATCHED THEN 
INSERT (employee_id,name,city,insert_timestamp,update_timestamp)
VALUES(source.employee_id,
source.name,
source.city,
current_timestamp(),
current_timestamp()
)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
select * from my_files.my_schema.emm_target_table

employee_id,name,city,insert_timestamp,update_timestamp
3,Charlie,Chicago,2026-03-07T12:10:49.292Z,2026-03-07T12:10:49.292Z
4,David,Houston,2026-03-07T12:10:49.292Z,2026-03-07T12:10:49.292Z
5,Eva,Phoenix,2026-03-07T12:10:49.292Z,2026-03-07T12:10:49.292Z
1,Alice,Phoenix,2026-03-07T12:10:49.292Z,2026-03-07T12:11:34.680Z
2,Bob,Houston,2026-03-07T12:10:49.292Z,2026-03-07T12:11:34.680Z
6,Alfred,Phoenix,2026-03-07T12:11:34.680Z,2026-03-07T12:11:34.680Z
7,Kate,Los Angeles,2026-03-07T12:11:34.680Z,2026-03-07T12:11:34.680Z


In [0]:
%sql
MERGE INTO my_files.my_schema.emm_target_table target using employee source on target.employee_id=source.employee_id

WHEN MATCHED THEN 
UPDATE SET 
target.name=source.name,
target.city=source.city,
target.update_timestamp=current_timestamp()
WHEN NOT MATCHED THEN 
INSERT (employee_id,name,city,insert_timestamp,update_timestamp)
VALUES(source.employee_id,
source.name,
source.city,
current_timestamp(),
current_timestamp()
) WHEN  NOT MATCHED BY SOURCE THEN DELETE

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
7,4,3,0


In [0]:
%sql
select * from my_files.my_schema.emm_target_table

employee_id,name,city,insert_timestamp,update_timestamp
7,Kate,Los Angeles,2026-03-07T12:11:34.680Z,2026-03-07T12:36:33.106Z
6,Alfred,Phoenix,2026-03-07T12:11:34.680Z,2026-03-07T12:36:33.106Z
1,Alice,Phoenix,2026-03-07T12:10:49.292Z,2026-03-07T12:36:33.106Z
2,Bob,Houston,2026-03-07T12:10:49.292Z,2026-03-07T12:36:33.106Z


In [0]:
from pyspark.sql import functions as F
df_employee.withColumn('insert_timestamp',F.current_timestamp()).withColumn('update_timestamp',F.current_timestamp()).createOrReplaceTempView('employee')

In [0]:
%sql
merge into my_files.my_schema.emm_target_table target using employee source
on target.employee_id=source.employee_id
WHEN MATCHED then
UPDATE SET *
WHEN NOT MATCHED then
INSERT *
WHEN NOT MATCHED BY SOURCE then
DELETE


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,4,0,0
